In [1]:
import numpy as np
from keras.preprocessing.image import ImageDataGenerator
from keras.datasets import cifar10
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers.core import Dense, Dropout, Activation, Flatten
from keras.layers.convolutional import Conv2D, MaxPooling2D
from keras.optimizers import SGD, Adam, RMSprop
import matplotlib.pyplot as plt


import ssl
ssl._create_default_https_context = ssl._create_unverified_context

#CIFAR_10 is a set of 60k images 32x32 pixels on 3 channels
IMG_CHANNELS = 3
IMG_ROWS = 32
IMG_COLS = 32

#constant
NUM_TO_AUGMENT=5
BATCH_SIZE = 128
NB_EPOCH = 25
NB_CLASSES = 10
VERBOSE = 1
VALIDATION_SPLIT = 0.2
OPTIM = RMSprop()

#load dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()


# Augmenting
print("Augmenting training set images...")
datagen = ImageDataGenerator( 
rotation_range=40,
width_shift_range=0.2,
height_shift_range=0.2,
zoom_range=0.2,
horizontal_flip=True,
fill_mode='nearest')

print('X_train shape:', X_train.shape)
print(X_train.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')

xtas, ytas = [], []
for i in range(X_train.shape[0]):
    num_aug = 0
    x = X_train[i] # (3, 32, 32)
    x = x.reshape((1,) + x.shape) # (1, 3, 32, 32)
    
for x_aug in datagen.flow(x, batch_size=1, save_to_dir='preview', save_prefix='cifar', save_format='jpeg'):
    if num_aug >= NUM_TO_AUGMENT:
        break
    xtas.append(x_aug[0])
    num_aug += 1

# convert to categorical
Y_train = np_utils.to_categorical(y_train, NB_CLASSES)
Y_test = np_utils.to_categorical(y_test, NB_CLASSES)

# float and normalization
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255

# network
model = Sequential()
model.add(Conv2D(32, (3, 3), padding='same', input_shape=(IMG_ROWS, IMG_COLS, IMG_CHANNELS)))
model.add(Activation('relu'))
model.add(Conv2D(32, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(Conv2D(64, 3, 3))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(NB_CLASSES))
model.add(Activation('softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', optimizer=OPTIM, metrics=['accuracy'])

#fit the dataget
datagen.fit(X_train)

#train
history = model.fit_generator(datagen.flow(X_train, Y_train, batch_size=BATCH_SIZE), samples_per_epoch=X_train.shape[0], epochs=NB_EPOCH, verbose=VERBOSE)
score = model.evaluate(X_test, Y_test,
                      batch_size=BATCH_SIZE, verbose=VERBOSE)
print("Test Score", score[0])
print('Test accuracy', score[1])

#save model
model_json = model.to_json()
open('cifar10_architecture.json', 'w').write(model_json)
# And the weights learned by our deep network on the training set
model.save_weights('cifar10_weights.h5', overwrite=True)

Using TensorFlow backend.


170500096/170498071 [==============================] - 2s 0us/step
Augmenting training set images...
X_train shape: (50000, 32, 32, 3)
50000 train samples
10000 test samples


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:79: UserWarning: Update your `Conv2D` call to the Keras 2 API: `Conv2D(64, (3, 3))`


Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 32, 32, 32)        896       
_________________________________________________________________
activation_1 (Activation)    (None, 32, 32, 32)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 32, 32, 32)        9248      
_________________________________________________________________
activation_2 (Activation)    (None, 32, 32, 32)        0         
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 16, 16, 32)        0         
_________________________________________________________________
dropout_1 (Dropout)          (None, 16, 16, 32)        0         
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 16, 16, 64)       

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:97: UserWarning: Update your `fit_generator` call to the Keras 2 API: `fit_generator(<keras.pre..., epochs=25, verbose=1, steps_per_epoch=390)`


Epoch 1/25
390/390 [==============================] - 131s 336ms/step - loss: 1.9491 - accuracy: 0.2865
Epoch 2/25
390/390 [==============================] - 146s 374ms/step - loss: 1.6656 - accuracy: 0.3957
Epoch 3/25
390/390 [==============================] - 141s 363ms/step - loss: 1.5305 - accuracy: 0.4490
Epoch 4/25
390/390 [==============================] - 141s 360ms/step - loss: 1.4476 - accuracy: 0.4821
Epoch 5/25
390/390 [==============================] - 152s 389ms/step - loss: 1.3882 - accuracy: 0.5017
Epoch 6/25
390/390 [==============================] - 151s 386ms/step - loss: 1.3402 - accuracy: 0.5222
Epoch 7/25
390/390 [==============================] - 150s 384ms/step - loss: 1.3115 - accuracy: 0.5334
Epoch 8/25
390/390 [==============================] - 165s 422ms/step - loss: 1.2813 - accuracy: 0.5469
Epoch 9/25
390/390 [==============================] - 163s 418ms/step - loss: 1.2579 - accuracy: 0.5553
Epoch 10/25
390/390 [==============================] - 138s 353m